# Battery Market Agent — 노드 · 그래프 · 워크플로우 시각화

현재까지 구현된 에이전트 그래프의 **구조, 노드, 데이터 흐름**을 시각화합니다.

| 섹션 | 내용 |
|------|------|
| 1 | LangGraph 네이티브 그래프 (SWOT 서브그래프 · 메인 그래프) |
| 2 | 전체 워크플로우 커스텀 다이어그램 |
| 3 | SWOT 서브그래프 상세 흐름 |
| 4 | State 스키마 (TypedDict 필드 테이블) |
| 5 | 구현 현황 요약 |

In [ ]:
import sys, os
import warnings
warnings.filterwarnings('ignore')

# 프로젝트 루트를 sys.path에 추가
for candidate in ['..', '.']:
    root = os.path.abspath(candidate)
    if os.path.exists(os.path.join(root, 'battery_market_agent')):
        if root not in sys.path:
            sys.path.insert(0, root)
        break

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import pandas as pd
from IPython.display import display, Image, Markdown

# 한글 폰트 설정
import matplotlib.font_manager as fm
korean_candidates = ['AppleGothic', 'NanumGothic', 'Malgun Gothic', 'NanumBarunGothic']
available_fonts = {f.name for f in fm.fontManager.ttflist}
for font_name in korean_candidates:
    if font_name in available_fonts:
        plt.rcParams['font.family'] = font_name
        break
plt.rcParams['axes.unicode_minus'] = False

print(f'sys.path root : {sys.path[0]}')
print(f'matplotlib font: {plt.rcParams["font.family"]}')

---
## 1. LangGraph 네이티브 그래프 시각화

`draw_mermaid_png()` 우선 시도 → 실패 시 Mermaid 텍스트로 폴백

In [ ]:
print('=== SWOT 서브그래프 ===')
try:
    from battery_market_agent.agents.swot.graph import build_swot_subgraph
    swot_graph = build_swot_subgraph()
    try:
        display(Image(swot_graph.get_graph().draw_mermaid_png()))
    except Exception:
        mermaid = swot_graph.get_graph().draw_mermaid()
        display(Markdown(f'```mermaid\n{mermaid}\n```'))
except Exception as e:
    print(f'SWOT 그래프 빌드 실패: {e}')

In [ ]:
print('=== 메인 그래프 (xray=True 로 서브그래프 포함) ===')
try:
    from battery_market_agent.agents.graph import build_graph
    main_graph = build_graph()
    try:
        display(Image(main_graph.get_graph(xray=True).draw_mermaid_png()))
    except Exception:
        mermaid = main_graph.get_graph(xray=True).draw_mermaid()
        display(Markdown(f'```mermaid\n{mermaid}\n```'))
        print('\nASCII fallback:')
        print(main_graph.get_graph().draw_ascii())
except Exception as e:
    print(f'메인 그래프 빌드 실패: {e}')

---
## 2. 전체 워크플로우 커스텀 다이어그램

구현 상태를 색상으로 표현합니다.

In [ ]:
C = {
    'done':       '#00897b',  # teal — 구현 완료
    'todo':       '#e53935',  # red  — 미구현
    'system':     '#37474f',  # dark — START/END
    'supervisor': '#5e35b1',  # deep purple — Supervisor
    'branch':     '#1e88e5',  # blue — 분기
}

def draw_node(ax, x, y, text, status, w=2.8, h=0.70):
    patch = FancyBboxPatch(
        (x - w/2, y - h/2), w, h,
        boxstyle='round,pad=0.12',
        facecolor=C[status], edgecolor='white',
        linewidth=2, alpha=0.93, zorder=4
    )
    ax.add_patch(patch)
    lines = text.split('\n')
    n = len(lines)
    for i, line in enumerate(lines):
        dy = (n - 1) * 0.13 - i * 0.26
        ax.text(x, y + dy, line, ha='center', va='center',
                color='white', fontsize=8.5, fontweight='bold', zorder=5)

def arr(ax, xs, ys, xe, ye, rad=0.0, lbl='', color='#546e7a', dashed=False, lw=1.8):
    ls = 'dashed' if dashed else 'solid'
    ax.annotate('', xy=(xe, ye), xytext=(xs, ys),
                arrowprops=dict(
                    arrowstyle='->', color=color, lw=lw,
                    linestyle=ls,
                    connectionstyle=f'arc3,rad={rad}'
                ), zorder=3)
    if lbl:
        mx = (xs + xe) / 2
        my = (ys + ye) / 2
        ax.text(mx + 0.10, my, lbl,
                ha='left', va='center', fontsize=6.5, color=color, style='italic')

def group_box(ax, x, y, w, h, lbl):
    rect = FancyBboxPatch(
        (x - w/2, y - h/2), w, h,
        boxstyle='round,pad=0.15',
        facecolor='#ede7f6', edgecolor='#7e57c2',
        linewidth=1.5, linestyle='--', alpha=0.30, zorder=2
    )
    ax.add_patch(rect)
    ax.text(x - w/2 + 0.15, y + h/2 - 0.05, lbl,
            ha='left', va='top', fontsize=7.5, color='#512da8', style='italic')

# ──────────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 19))
fig.patch.set_facecolor('#f5f5f5')
ax.set_facecolor('#f5f5f5')
ax.set_xlim(-0.5, 13.5)
ax.set_ylim(3.5, 17.5)
ax.axis('off')
ax.set_title('Battery Market Agent — 전체 워크플로우',
             fontsize=15, fontweight='bold', pad=20, color='#212121')

# ── START → retrieve → branch ─────────────────────────────────────────────────
draw_node(ax, 6.5, 17.0, 'START', 'system', 1.6, 0.6)
arr(ax, 6.5, 16.70, 6.5, 16.10)
draw_node(ax, 6.5, 15.75, 'retrieve\n(RAG 문서 검색)', 'todo')
arr(ax, 6.5, 15.40, 6.5, 14.75)
draw_node(ax, 6.5, 14.40, 'branch_companies\n(Send API — 병렬 분기)', 'branch', 3.4, 0.70)

# 병렬 분기 화살표
arr(ax, 5.0, 14.05, 3.5, 13.35, rad=-0.15, lbl='LG에너지솔루션')
arr(ax, 8.0, 14.05, 9.5, 13.35, rad= 0.15, lbl='CATL')

# ── LG 그룹 ───────────────────────────────────────────────────────────────────
group_box(ax, 3.5, 11.85, 5.8, 3.10, 'company_analysis [LG에너지솔루션]')
draw_node(ax, 3.5, 13.00, 'company_analysis\n[LG에너지솔루션] ✅', 'supervisor', 3.2, 0.70)

# 위임: supervisor → sub-agents (dashed)
arr(ax, 2.6, 12.65, 1.9, 12.15, dashed=True, color='#7e57c2', lbl='위임')
arr(ax, 3.5, 12.65, 3.5, 12.15, dashed=True, color='#7e57c2')
arr(ax, 4.4, 12.65, 5.1, 12.15, dashed=True, color='#7e57c2')

draw_node(ax, 1.8, 11.75, 'market_analysis\n✅', 'done', 2.2, 0.65)
draw_node(ax, 3.5, 11.75, 'tech_analysis\n✅',   'done', 2.0, 0.65)
draw_node(ax, 5.2, 11.75, 'swot_analysis\n✅',   'done', 2.0, 0.65)

# 반환: sub-agents → supervisor (curved, teal)
arr(ax, 2.1, 12.07, 2.9, 12.65, rad=-0.35, color='#00897b', lbl='반환')
arr(ax, 3.5, 12.07, 3.5, 12.65, rad= 0.35, color='#00897b')
arr(ax, 4.9, 12.07, 4.1, 12.65, rad=-0.35, color='#00897b')

# supervisor → company_comparison
arr(ax, 3.5, 12.65, 5.4, 10.40, rad=0.15, color='#546e7a')

# ── CATL 그룹 ─────────────────────────────────────────────────────────────────
group_box(ax, 9.5, 11.85, 5.8, 3.10, 'company_analysis [CATL]')
draw_node(ax, 9.5, 13.00, 'company_analysis\n[CATL] ✅', 'supervisor', 3.0, 0.70)

# 위임: supervisor → sub-agents (dashed)
arr(ax,  8.6, 12.65,  7.9, 12.15, dashed=True, color='#7e57c2', lbl='위임')
arr(ax,  9.5, 12.65,  9.5, 12.15, dashed=True, color='#7e57c2')
arr(ax, 10.4, 12.65, 11.1, 12.15, dashed=True, color='#7e57c2')

draw_node(ax,  7.8, 11.75, 'market_analysis\n✅', 'done', 2.2, 0.65)
draw_node(ax,  9.5, 11.75, 'tech_analysis\n✅',   'done', 2.0, 0.65)
draw_node(ax, 11.2, 11.75, 'swot_analysis\n✅',   'done', 2.0, 0.65)

# 반환: sub-agents → supervisor (curved, teal)
arr(ax,  8.1, 12.07,  8.9, 12.65, rad= 0.35, color='#00897b', lbl='반환')
arr(ax,  9.5, 12.07,  9.5, 12.65, rad=-0.35, color='#00897b')
arr(ax, 10.9, 12.07, 10.1, 12.65, rad= 0.35, color='#00897b')

# supervisor → company_comparison
arr(ax, 9.5, 12.65, 7.6, 10.40, rad=-0.15, color='#546e7a')

# ── 합류 → 비교 → 보고서 ──────────────────────────────────────────────────────
draw_node(ax, 6.5, 10.05, 'company_comparison', 'todo')
arr(ax, 6.5, 9.70, 6.5, 9.00)
draw_node(ax, 6.5, 8.65, 'report_generation ✅', 'done')
arr(ax, 6.5, 8.30, 6.5, 7.65)
draw_node(ax, 6.5, 7.30, 'END', 'system', 1.6, 0.60)

# ── 범례 ───────────────────────────────────────────────────────────────────────
handles = [
    mpatches.Patch(facecolor=C['done'],       label='✅ 구현 완료'),
    mpatches.Patch(facecolor=C['todo'],       label='❌ 미구현'),
    mpatches.Patch(facecolor=C['branch'],     label='🔀 분기 (Send API)'),
    mpatches.Patch(facecolor=C['supervisor'], label='🎯 Supervisor ✅ 구현'),
    mpatches.Patch(facecolor=C['system'],     label='⬛ 시스템 노드'),
    mpatches.Patch(facecolor='#7e57c2',       label='- - 위임 (Supervisor → Sub-agent)'),
    mpatches.Patch(facecolor='#00897b',       label='↩ 반환 (Sub-agent → Supervisor)'),
]
ax.legend(handles=handles, loc='lower right', fontsize=9,
          framealpha=0.95, facecolor='white', edgecolor='#bdbdbd')

plt.tight_layout()
plt.savefig('workflow_diagram.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. SWOT 서브그래프 상세 흐름

In [ ]:
fig2, ax2 = plt.subplots(figsize=(11, 9))
fig2.patch.set_facecolor('#f5f5f5')
ax2.set_facecolor('#f5f5f5')
ax2.set_xlim(0, 11)
ax2.set_ylim(-0.5, 9.5)
ax2.axis('off')
ax2.set_title('SWOT 서브그래프 — 노드 상세 흐름',
              fontsize=13, fontweight='bold', pad=16)

NODE_COLOR = '#00897b'
STATE_BG   = '#e3f2fd'
STATE_BD   = '#90caf9'

def snode(x, y, title, subtitle, w=5.5, h=1.0):
    patch = FancyBboxPatch(
        (x - w/2, y - h/2), w, h,
        boxstyle='round,pad=0.15',
        facecolor=NODE_COLOR, edgecolor='white',
        linewidth=2, alpha=0.93, zorder=4
    )
    ax2.add_patch(patch)
    ax2.text(x, y + 0.16, title, ha='center', va='center',
             color='white', fontsize=10, fontweight='bold', zorder=5)
    ax2.text(x, y - 0.20, subtitle, ha='center', va='center',
             color='#b2dfdb', fontsize=8, zorder=5)

def sarr(xs, ys, xe, ye):
    ax2.annotate('', xy=(xe, ye), xytext=(xs, ys),
                 arrowprops=dict(arrowstyle='->', color='#546e7a', lw=2),
                 zorder=3)

def state_badge(x, y, text):
    ax2.text(x, y, text, ha='center', va='center', fontsize=8,
             color='#1565c0',
             bbox=dict(boxstyle='round,pad=0.35',
                       facecolor=STATE_BG, edgecolor=STATE_BD,
                       alpha=0.90))

def sys_node(x, y, text, w=2.0, h=0.55):
    patch = FancyBboxPatch(
        (x - w/2, y - h/2), w, h,
        boxstyle='round,pad=0.10',
        facecolor='#37474f', edgecolor='white',
        linewidth=2, alpha=0.93, zorder=4
    )
    ax2.add_patch(patch)
    ax2.text(x, y, text, ha='center', va='center',
             color='white', fontsize=9, fontweight='bold', zorder=5)

# 입력 상태
ax2.text(5.5, 9.0,
         'SWOTState 입력:  subject(str)  raw_info=[]  criteria={}',
         ha='center', va='center', fontsize=8.5, color='#37474f',
         bbox=dict(boxstyle='round,pad=0.4',
                   facecolor='#fff9c4', edgecolor='#f9a825', alpha=0.9))

sarr(5.5, 8.72, 5.5, 8.10)

# 노드 1
snode(5.5, 7.65, 'gather_info_node',
      'fetch_google_news + search_web → 원시 정보 수집')
state_badge(8.8, 7.65, 'raw_info: list[str]\n(Annotated add 누적)')
sarr(5.5, 7.15, 5.5, 6.50)

# 노드 2
snode(5.5, 6.05, 'classify_swot_node',
      'LLM structured output (SWOTItems) → S/W/O/T 항목 분류')
state_badge(8.8, 6.05, 'strengths / weaknesses\nopportunities / threats')
sarr(5.5, 5.55, 5.5, 4.90)

# 노드 3
snode(5.5, 4.45, 'format_matrix_node',
      'analyze_swot 도구 호출 → 2×2 SWOT 행렬 렌더링')
state_badge(8.8, 4.45, 'swot_matrix: str\n(2×2 텍스트 테이블)')
sarr(5.5, 3.95, 5.5, 3.35)

sys_node(5.5, 3.08, 'END')
sarr(5.5, 2.80, 5.5, 2.20)

# 출력 상태
ax2.text(5.5, 1.85,
         'SWOTState 출력:  strengths  weaknesses  opportunities  threats  swot_matrix',
         ha='center', va='center', fontsize=8.5, color='#37474f',
         bbox=dict(boxstyle='round,pad=0.4',
                   facecolor='#f3e5f5', edgecolor='#ab47bc', alpha=0.90))

# 상태 흐름 레이블
ax2.text(8.8, 8.40, '← State 변경 필드', ha='center', va='center',
         fontsize=8, color='#9e9e9e', style='italic')

plt.tight_layout()
plt.savefig('swot_subgraph.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. State 스키마 — TypedDict 필드 테이블

In [ ]:
pd.set_option('display.max_colwidth', 60)

states = {
    'MarketAnalysisState\n(agents/market/state.py)': [
        ('company',                  'str',                        '입력',       '분석 대상 회사명'),
        ('market_size_raw',          'str',                        '웹 검색',    '글로벌 시장 규모·성장률'),
        ('regional_demand_raw',      'str',                        '웹 검색',    '지역별 수요 동향'),
        ('raw_material_prices_raw',  'str',                        '웹 검색',    '원자재(Li·Co·Ni) 가격'),
        ('company_market_share_raw', 'str',                        '웹 검색',    '기업 시장점유율'),
        ('battery_price_per_kwh_raw','str',                        '웹 검색',    '배터리 팩 단가 추이($/kWh)'),
        ('regulatory_policy_raw',    'str',                        '웹 검색',    'EU 규제·IRA 정책'),
        ('news_items',               'Annotated[list[str], add]',  '웹 검색',    '최신 뉴스 목록 (누적)'),
        ('market_analysis',          'str',                        '최종 결과',  'LLM 종합 시장 분석 보고서'),
    ],
    'TechAnalysisState\n(state/tech_analysis_state.py)': [
        ('company',       'str',                           '입력',    '분석 대상 회사명'),
        ('tech_queries',  'list[str]',                     '입력',    'RAG 검색 쿼리 목록'),
        ('retrieved_docs','Annotated[list[Document], add]','중간',    '쿼리별 RAG 결과 (누적)'),
        ('tech_analysis', 'str',                           '최종',    '최종 기술 역량 분석 보고서'),
    ],
    'SWOTState\n(agents/swot/state.py)': [
        ('subject',      'str',                        '입력',  '분석 대상 (기업명 등)'),
        ('raw_info',     'Annotated[list[str], add]',  '중간',  '원시 정보 누적'),
        ('criteria',     'dict[str, str]',             '입력',  'S/W/O/T별 평가 기준 (선택)'),
        ('strengths',    'list[str]',                  '중간',  '내부 강점 목록'),
        ('weaknesses',   'list[str]',                  '중간',  '내부 약점 목록'),
        ('opportunities','list[str]',                  '중간',  '외부 기회 목록'),
        ('threats',      'list[str]',                  '중간',  '외부 위협 목록'),
        ('swot_matrix',  'str',                        '최종',  '2×2 SWOT 행렬 문자열'),
    ],
    'CompanyAnalysisState\n(state/company_analysis_state.py)': [
        ('company',        'str',                             '입력',         '분석 대상 회사명'),
        ('retrieved_docs', 'list[Document]',                  '입력',         '상위 그래프 RAG 결과'),
        ('messages',       'Annotated[list[AnyMessage], ...]','Supervisor',   'LLM 대화 히스토리'),
        ('market_analysis','str',                             '하위 에이전트', 'market_analysis_agent 결과'),
        ('tech_analysis',  'str',                             '하위 에이전트', 'tech_analysis_agent 결과'),
        ('strengths',      'list[str]',                       '하위 에이전트', 'SWOT 강점'),
        ('weaknesses',     'list[str]',                       '하위 에이전트', 'SWOT 약점'),
        ('opportunities',  'list[str]',                       '하위 에이전트', 'SWOT 기회'),
        ('threats',        'list[str]',                       '하위 에이전트', 'SWOT 위협'),
        ('swot_matrix',    'str',                             '하위 에이전트', '2×2 SWOT 행렬'),
        ('company_report', 'str',                             '최종 출력',     'Supervisor 종합 보고서'),
    ],
    'ReportState\n(state/report_state.py)': [
        ('company_report',    'dict[str, str]',       '입력',    '기업 분석 결과 {"LG...": ..., "CATL": ...}'),
        ('comparison_report', 'str',                  '입력',    '회사 비교 에이전트 결과'),
        ('sections',          'ReportSections | None','중간',    'LLM structured output 섹션 데이터'),
        ('final_report',      'str',                  '최종 결과','마크다운 렌더링된 최종 보고서'),
    ],
}

for title, fields in states.items():
    df = pd.DataFrame(fields, columns=['필드명', '타입', '구분', '설명'])
    clean_title = title.replace('\n', '  ')
    print(f'\n{"━"*72}')
    print(f'  {clean_title}')
    print('━'*72)
    display(
        df.style
          .set_properties(**{'text-align': 'left', 'font-size': '12px'})
          .map(lambda v: 'background-color:#e8f5e9' if v in ('최종 결과', '최종')
                    else ('background-color:#e3f2fd' if v in ('웹 검색', '중간')
                    else ('background-color:#fff9c4' if v == '입력'
                    else ('background-color:#f3e5f5' if v in ('Supervisor', '하위 에이전트')
                    else ''))), subset=['구분'])
          .hide(axis='index')
    )

---
## 5. 구현 현황 요약

In [ ]:
items = [
    # (label, done, category)
    ('market_analysis_agent',        True,  '에이전트'),
    ('tech_analysis_agent',          True,  '에이전트'),
    ('swot_analysis_agent',          True,  '에이전트'),
    ('company_analysis_agent',       True,  '에이전트'),
    ('company_comparison_agent',     False, '에이전트'),
    ('report_generation_agent',      True,  '에이전트'),
    ('SWOT 서브그래프 (3 노드)',      True,  '서브그래프'),
    ('보고서 서브그래프 (2 노드)',    True,  '서브그래프'),
    ('retrieve_node',                False, '서브그래프'),
    ('MarketAnalysisState',          True,  'State'),
    ('TechAnalysisState',            True,  'State'),
    ('SWOTState',                    True,  'State'),
    ('CompanyAnalysisState',         True,  'State'),
    ('ReportState / ReportSections', True,  'State'),
    ('BatteryMarketState',           False, 'State'),
    ('search_web',                   True,  '도구'),
    ('fetch_google_news',            True,  '도구'),
    ('read_pdf',                     True,  '도구'),
    ('analyze_swot',                 True,  '도구'),
    ('fetch_price_trends',           False, '도구'),
    ('search_battery_market_data',   False, '도구'),
    ('summarize_regulations',        False, '도구'),
    ('BatteryRAG',                   True,  'RAG'),
]

labels = [i[0] for i in items]
done   = [1 if i[1] else 0 for i in items]
cats   = [i[2] for i in items]

cat_colors = {
    '에이전트':   '#5e35b1',
    '서브그래프': '#1e88e5',
    'State':      '#00897b',
    '도구':       '#f57c00',
    'RAG':        '#37474f',
}
bar_colors = [
    cat_colors[c] if d else '#ef9a9a'
    for c, d in zip(cats, done)
]

fig3, ax3 = plt.subplots(figsize=(11, 11))
fig3.patch.set_facecolor('#f5f5f5')
ax3.set_facecolor('#f5f5f5')

y_pos = range(len(labels))
bars = ax3.barh(list(y_pos), done, color=bar_colors,
                edgecolor='white', linewidth=1.5, height=0.65)

# 레이블
for i, (bar, label, d) in enumerate(zip(bars, labels, done)):
    mark = '✅' if d else '❌'
    ax3.text(-0.02, i, f'{mark}  {label}',
             ha='right', va='center', fontsize=9.5, color='#212121')

# 카테고리 레이블 (오른쪽)
for i, cat in enumerate(cats):
    ax3.text(1.02, i, cat, ha='left', va='center',
             fontsize=8.5, color=cat_colors[cat], fontweight='bold')

# 구현율
total_done = sum(done)
total = len(done)
ax3.set_title(
    f'구현 현황  —  {total_done}/{total} 완료  ({total_done/total*100:.0f}%)',
    fontsize=13, fontweight='bold', pad=16, color='#212121'
)

ax3.set_xlim(-5.5, 1.35)
ax3.set_yticks([])
ax3.set_xticks([])
ax3.axvline(0, color='#bdbdbd', lw=1)
ax3.invert_yaxis()

# 범례
legend_handles = [
    mpatches.Patch(facecolor=c, label=cat)
    for cat, c in cat_colors.items()
] + [mpatches.Patch(facecolor='#ef9a9a', label='❌ 미구현')]
ax3.legend(handles=legend_handles, loc='lower right',
           fontsize=9, framealpha=0.95, facecolor='white')

plt.tight_layout()
plt.savefig('impl_status.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n구현 완료: {total_done}/{total} ({total_done/total*100:.0f}%)')